<a href="https://colab.research.google.com/github/mitalidaduria/payment-intelligence-platform/blob/main/src_3_Layer_Data_Quality_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from dataclasses import dataclass, field
from typing import List, Dict
import pandas as pd

@dataclass
class DQResult:
    layer: str
    rule: str
    passed: bool
    failing_rows: int = 0
    detail: str = ""

class PaymentDQFramework:
    """3-Layer DQ Governance — mirrors enterprise PCI-DSS implementation."""

    # Layer 1: Definition Control
    def layer1_definition_control(self, df: pd.DataFrame) -> List[DQResult]:
        results = []
        required = ["txn_id", "amount", "gateway", "status", "ts"]
        for col in required:
            null_count = df[col].isna().sum()
            results.append(DQResult(
                layer="L1-DefinitionControl",
                rule=f"not_null:{col}",
                passed=null_count == 0,
                failing_rows=null_count,
                detail=f"{null_count} nulls in {col}"
            ))
        return results

    # Layer 2: Rule Enforcement
    def layer2_rule_enforcement(self, df: pd.DataFrame) -> List[DQResult]:
        results = []
        invalid_amt = (df["amount"] <= 0).sum()
        results.append(DQResult("L2-RuleEnforcement", "amount_positive", invalid_amt == 0, invalid_amt))

        invalid_gw = ~df["gateway"].isin(["razorpay", "payu", "stripe"])
        results.append(DQResult("L2-RuleEnforcement", "gateway_valid", invalid_gw.sum() == 0, invalid_gw.sum()))
        return results

    # Layer 3: Operational Alerting
    def layer3_operational_alert(self, df: pd.DataFrame, failure_threshold: float = 0.15) -> DQResult:
        failure_rate = (df["status"] == "FAILED").mean()
        return DQResult(
            layer="L3-OperationalAlert",
            rule="failure_rate_threshold",
            passed=failure_rate <= failure_threshold,
            detail=f"Failure rate: {failure_rate:.1%} (threshold {failure_threshold:.0%})"
        )

    def run_all(self, df: pd.DataFrame) -> Dict:
        all_r = self.layer1_definition_control(df) + self.layer2_rule_enforcement(df) + [self.layer3_operational_alert(df)]
        passed = sum(1 for r in all_r if r.passed)
        return {
            "total": len(all_r),
            "passed": passed,
            "dq_score": f"{passed/len(all_r):.1%}",
            "results": [vars(r) for r in all_r]
        }